# MESC Experiment-0 — Colab Foundation Tournament

**Protocol status:** preparation template only.

This notebook is a fail-closed execution surface for the future no-training MESC foundation tournament.
It is not runtime evidence until run under a canonical frozen `MESC-EXPERIMENT-0-CONFIG-V1`
whose MRL-0801..MRL-0899 authority/evidence bindings are genuine and current.

Do not edit cells to bypass a blocked gate. Do not paste credentials into source or output.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import subprocess
from datetime import datetime, timezone
from pathlib import Path

CONFIG_PATH = Path("/content/experiment-config.json")
ROOT = Path("/content/mesc-experiment-0")
REPO = ROOT / "repo"
EVIDENCE = ROOT / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=True)

CONFIG_SCHEMA = "MESC-EXPERIMENT-0-CONFIG-V1"
RUNTIME_SCHEMA = "MESC-EXPERIMENT-0-RUNTIME-V1"
ENVIRONMENT_SCHEMA = "MESC-EXPERIMENT-0-ENVIRONMENT-V1"
GIT_RE = re.compile(r"^[0-9a-f]{40}$")
KEY_RE = re.compile(r"^[a-z0-9][a-z0-9._-]{0,63}$")
CANDIDATE_CLASSES = {"SELECTABLE_FOUNDATION", "REFERENCE_ONLY"}
SECRET_FIELDS = {
    "token",
    "hf_token",
    "access_token",
    "refresh_token",
    "password",
    "secret",
    "api_key",
    "apikey",
    "private_key",
    "client_secret",
    "authorization_header",
}
SECRET_PATTERNS = (
    re.compile(r"\bhf_[A-Za-z0-9]{20,}\b"),
    re.compile(r"\bgh[pousr]_[A-Za-z0-9_]{20,}\b"),
    re.compile(r"\bgithub_pat_[A-Za-z0-9_]{20,}\b"),
    re.compile(r"\bsk-[A-Za-z0-9_-]{20,}\b"),
    re.compile(r"(?i)\bauthorization\s*:\s*bearer\s+\S+"),
    re.compile(r"https?://[^/\s:@]+:[^@\s/]+@"),
)
REQUIRED_AUTHORITY_KEYS = (
    "mrl_0801_evidence_id",
    "mrl_0802_evidence_id",
    "mrl_0803_evidence_id",
    "mrl_0804_evidence_id",
    "mrl_0805_authority_id",
    "mrl_0806_objective_id",
    "mrl_0807_evaluator_freeze_id",
    "mrl_0808_sandbox_id",
    "mrl_0809_preflight_id",
    "mrl_0899_readiness_id",
)
REQUIRED_FROZEN_FIELDS = (
    "experiment_id",
    "objective_id",
    "repository_sha",
    "repository_tree",
    "strategy_decision_id",
    "candidate_roster",
    "dataset_identities",
    "evaluator_identities",
    "scoring_policy_identities",
    "prompt_template_identities",
    "generation_configs",
    "runtime_policy",
    "network_policy",
    "filesystem_policy",
    "credential_policy",
    "resource_budget",
    "query_budget",
    "result_exposure_budget",
    "hard_floor_policy",
    "decision_rule",
    "sealed_evaluation_policy",
    "authority_bindings",
)
FROZEN_IDENTITY_REQUIREMENTS = {
    "dataset_identities": ("dataset_id", "split_id", "held_out_tier"),
    "evaluator_identities": ("evaluator_id",),
    "scoring_policy_identities": ("scoring_policy_id",),
    "prompt_template_identities": ("prompt_template_id",),
    "generation_configs": ("generation_config_id",),
}


def canonical_json_bytes(value: object) -> bytes:
    """Serialize deterministic UTF-8 JSON bytes for hashing and evidence files."""
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def sha256_bytes(data: bytes) -> str:
    """Return the lowercase SHA-256 digest of exact bytes."""
    return hashlib.sha256(data).hexdigest()


def write_json(path: Path, value: object) -> str:
    """Write exact canonical JSON bytes and return the digest of those stored bytes."""
    data = canonical_json_bytes(value)
    path.write_bytes(data)
    return sha256_bytes(data)


def blocked(reason: str, **facts: object) -> None:
    """Persist a metadata-only blocked receipt and stop execution."""
    payload = {
        "schema_version": "MESC-EXPERIMENT-0-BLOCKED-V1",
        "reason": reason,
        "facts": facts,
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(EVIDENCE / "blocked.json", payload)
    raise RuntimeError(f"MESC Experiment-0 BLOCKED: {reason}")


def strict_json_loads(text: str) -> object:
    """Reject duplicate keys and non-finite constants before config semantics run."""

    def reject_pairs(pairs: list[tuple[str, object]]) -> dict[str, object]:
        value: dict[str, object] = {}
        for key, child in pairs:
            if key in value:
                raise ValueError(f"duplicate JSON key: {key}")
            value[key] = child
        return value

    def reject_constant(value: str) -> None:
        raise ValueError(f"non-finite JSON constant: {value}")

    return json.loads(
        text,
        object_pairs_hook=reject_pairs,
        parse_constant=reject_constant,
    )


def validate_no_secret_bearing_config(text: str, value: object) -> None:
    """Reject serialized secrets and secret-like field names before runtime access."""
    if any(pattern.search(text) for pattern in SECRET_PATTERNS):
        raise ValueError("possible secret-bearing value detected")

    def scan(child: object, path: str) -> None:
        if isinstance(child, dict):
            for key, nested in child.items():
                if str(key).strip().lower() in SECRET_FIELDS:
                    raise ValueError(f"{path}: forbidden secret-bearing field {key!r}")
                scan(nested, f"{path}.{key}")
        elif isinstance(child, list):
            for index, nested in enumerate(child):
                scan(nested, f"{path}[{index}]")

    scan(value, "experiment-config")


def validate_git_identity(value: object, label: str) -> None:
    """Require an immutable lowercase 40-hex Git object identity."""
    if not isinstance(value, str) or not GIT_RE.fullmatch(value):
        raise ValueError(f"{label}: expected lowercase 40-hex Git identity")


def validate_candidate_roster(records: object) -> None:
    """Reject ambiguous or non-immutable candidate identities before network access."""
    if not isinstance(records, list) or not records:
        raise ValueError("candidate_roster: expected non-empty list")
    identities: set[tuple[str, str]] = set()
    evidence_keys: set[str] = set()
    for index, record in enumerate(records):
        label = f"candidate_roster[{index}]"
        if not isinstance(record, dict):
            raise ValueError(f"{label}: expected object")
        candidate_id = record.get("candidate_id")
        revision = record.get("candidate_revision")
        candidate_class = record.get("candidate_class")
        evidence_key = record.get("evidence_key")
        modalities = record.get("supported_input_modalities")
        if not isinstance(candidate_id, str) or not candidate_id.strip():
            raise ValueError(f"{label}.candidate_id: expected non-empty string")
        validate_git_identity(revision, f"{label}.candidate_revision")
        assert isinstance(revision, str)
        if candidate_class not in CANDIDATE_CLASSES:
            raise ValueError(f"{label}.candidate_class: invalid class")
        if not isinstance(evidence_key, str) or not KEY_RE.fullmatch(evidence_key):
            raise ValueError(f"{label}.evidence_key: invalid key")
        if not isinstance(modalities, list) or "text" not in modalities:
            raise ValueError(f"{label}.supported_input_modalities: text is required")
        if candidate_class == "SELECTABLE_FOUNDATION" and "vision" not in modalities:
            raise ValueError(f"{label}: selectable foundation requires vision")
        identity = (candidate_id, revision)
        if identity in identities or evidence_key in evidence_keys:
            raise ValueError(f"{label}: duplicate candidate identity")
        identities.add(identity)
        evidence_keys.add(evidence_key)


def validate_frozen_identity_records(
    records: object,
    key_fields: tuple[str, ...],
    label: str,
) -> None:
    """Reject malformed, blank, or duplicate frozen evaluation identities."""
    if not isinstance(records, list) or not records:
        raise ValueError(f"{label}: expected non-empty list")
    seen: set[tuple[str, ...]] = set()
    for index, record in enumerate(records):
        if not isinstance(record, dict):
            raise ValueError(f"{label}[{index}]: expected object")
        identity: list[str] = []
        for field in key_fields:
            value = record.get(field)
            if not isinstance(value, str) or not value.strip():
                raise ValueError(f"{label}[{index}].{field}: expected non-empty string")
            identity.append(value)
        identity_key = tuple(identity)
        if identity_key in seen:
            raise ValueError(f"{label}: duplicate frozen identity")
        seen.add(identity_key)


def validate_non_negative_finite_number(value: object, label: str) -> None:
    """Require a numeric frozen budget that is finite and zero or greater."""
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value < 0:
        raise ValueError(f"{label}: expected non-negative finite number")
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError(f"{label}: expected non-negative finite number")


def validate_budget_config(config: dict[str, object]) -> None:
    """Validate all numeric Experiment-0 budget ceilings before runtime access."""
    resource_budget = config["resource_budget"]
    query_budget = config["query_budget"]
    result_exposure_budget = config["result_exposure_budget"]
    assert isinstance(resource_budget, dict)
    assert isinstance(query_budget, dict)
    assert isinstance(result_exposure_budget, dict)
    for key in ("max_gpu_hours", "max_wall_hours", "max_storage_bytes", "max_retries"):
        validate_non_negative_finite_number(resource_budget.get(key), f"resource_budget.{key}")
    validate_non_negative_finite_number(
        query_budget.get("max_adaptive_queries"),
        "query_budget.max_adaptive_queries",
    )
    for key in ("tier1_max_exposures", "tier2_max_exposures"):
        validate_non_negative_finite_number(
            result_exposure_budget.get(key),
            f"result_exposure_budget.{key}",
        )
    if result_exposure_budget.get("tier3_allowed_fields") is None:
        raise ValueError("result_exposure_budget.tier3_allowed_fields: explicit value required")


def validate_runtime_policy(policy: object) -> None:
    """Require the exact hosted-GPU policy shape enforced by the evidence verifier."""
    if not isinstance(policy, dict):
        raise ValueError("runtime_policy: expected object")
    if policy.get("provider") != "GOOGLE_COLAB":
        raise ValueError("runtime_policy.provider: expected GOOGLE_COLAB")
    if policy.get("require_hosted_gpu") is not True:
        raise ValueError("runtime_policy.require_hosted_gpu: expected true")
    allowed_gpu_count = policy.get("allowed_gpu_count")
    if isinstance(allowed_gpu_count, bool) or not isinstance(allowed_gpu_count, int) or allowed_gpu_count < 1:
        raise ValueError("runtime_policy.allowed_gpu_count: expected positive integer")
    allowed_gpu_models = policy.get("allowed_gpu_models")
    if not isinstance(allowed_gpu_models, list) or any(
        not isinstance(name, str) or not name.strip() for name in allowed_gpu_models
    ):
        raise ValueError("runtime_policy.allowed_gpu_models: invalid model list")
    if len(allowed_gpu_models) != len(set(allowed_gpu_models)):
        raise ValueError("runtime_policy.allowed_gpu_models: duplicate model name")
    allow_unlisted = policy.get("allow_unlisted_gpu_model")
    if not isinstance(allow_unlisted, bool):
        raise ValueError("runtime_policy.allow_unlisted_gpu_model: expected boolean")
    if not allow_unlisted and not allowed_gpu_models:
        raise ValueError("runtime_policy.allowed_gpu_models: no admissible GPU")


In [ ]:
# Load and validate the exact frozen config before any network/model access.
if not CONFIG_PATH.exists():
    blocked("FROZEN_CONFIG_MISSING", expected_path=str(CONFIG_PATH))

try:
    config_text = CONFIG_PATH.read_bytes().decode("utf-8")
    config = strict_json_loads(config_text)
    validate_no_secret_bearing_config(config_text, config)
except (UnicodeDecodeError, json.JSONDecodeError, ValueError, RecursionError) as exc:
    blocked("FROZEN_CONFIG_INVALID_JSON", failure_class=type(exc).__name__)

if not isinstance(config, dict):
    blocked("FROZEN_CONFIG_NOT_OBJECT")
if config.get("schema_version") != CONFIG_SCHEMA:
    blocked("CONFIG_SCHEMA_MISMATCH", observed=config.get("schema_version"))
if config.get("status") != "FROZEN_EXECUTION_CONFIG":
    blocked("CONFIG_NOT_FROZEN", observed=config.get("status"))

missing_fields = [field for field in REQUIRED_FROZEN_FIELDS if field not in config]
if missing_fields:
    blocked("CONFIG_REQUIRED_FIELD_MISSING", fields=missing_fields)

for field in ("experiment_id", "objective_id", "strategy_decision_id"):
    if not isinstance(config.get(field), str) or not config[field].strip():
        blocked("CONFIG_REQUIRED_FIELD_INVALID", field=field)

for field in ("repository_sha", "repository_tree"):
    try:
        validate_git_identity(config.get(field), field)
    except ValueError as exc:
        blocked(
            "CONFIG_REPOSITORY_IDENTITY_INVALID",
            field=field,
            failure_message_sha256=sha256_bytes(str(exc).encode("utf-8")),
        )

try:
    validate_candidate_roster(config.get("candidate_roster"))
except ValueError as exc:
    blocked(
        "CONFIG_CANDIDATE_ROSTER_INVALID",
        failure_message_sha256=sha256_bytes(str(exc).encode("utf-8")),
    )

for field in FROZEN_IDENTITY_REQUIREMENTS:
    value = config.get(field)
    if not isinstance(value, list) or not value:
        blocked("CONFIG_FROZEN_LIST_INVALID", field=field)

for field, key_fields in FROZEN_IDENTITY_REQUIREMENTS.items():
    try:
        validate_frozen_identity_records(config[field], key_fields, field)
    except ValueError as exc:
        blocked(
            "CONFIG_FROZEN_IDENTITY_INVALID",
            field=field,
            failure_message_sha256=sha256_bytes(str(exc).encode("utf-8")),
        )

for field in (
    "runtime_policy",
    "network_policy",
    "filesystem_policy",
    "credential_policy",
    "resource_budget",
    "query_budget",
    "result_exposure_budget",
    "hard_floor_policy",
    "decision_rule",
    "sealed_evaluation_policy",
    "authority_bindings",
):
    value = config.get(field)
    if not isinstance(value, dict) or not value:
        blocked("CONFIG_FROZEN_OBJECT_INVALID", field=field)

bindings = config["authority_bindings"]
missing_bindings = [key for key in REQUIRED_AUTHORITY_KEYS if not bindings.get(key)]
if missing_bindings:
    blocked("MRL_AUTHORITY_OR_EVIDENCE_BINDING_MISSING", missing=missing_bindings)

try:
    validate_budget_config(config)
except ValueError as exc:
    blocked(
        "FROZEN_BUDGET_INVALID",
        failure_message_sha256=sha256_bytes(str(exc).encode("utf-8")),
    )

sealed_policy = config["sealed_evaluation_policy"]
if sealed_policy.get("tier3_item_access_by_research_process") is not False:
    blocked("SEALED_TIER3_POLICY_INVALID")

runtime_policy = config.get("runtime_policy")
if not isinstance(runtime_policy, dict):
    blocked("RUNTIME_POLICY_MISSING_OR_INVALID")
if runtime_policy.get("provider") != "GOOGLE_COLAB":
    blocked("RUNTIME_POLICY_PROVIDER_INVALID", observed=runtime_policy.get("provider"))
allowed_gpu_count = runtime_policy.get("allowed_gpu_count")
if (
    isinstance(allowed_gpu_count, bool)
    or not isinstance(allowed_gpu_count, int)
    or allowed_gpu_count < 1
):
    blocked("RUNTIME_POLICY_GPU_COUNT_INVALID", observed=allowed_gpu_count)
try:
    validate_runtime_policy(runtime_policy)
except ValueError as exc:
    blocked(
        "RUNTIME_POLICY_INVALID",
        failure_message_sha256=sha256_bytes(str(exc).encode("utf-8")),
    )

config_sha256 = sha256_bytes(canonical_json_bytes(config))
print(json.dumps({
    "config_schema": CONFIG_SCHEMA,
    "experiment_id": config["experiment_id"],
    "config_sha256": config_sha256,
    "candidate_count": len(config["candidate_roster"]),
}, indent=2))


In [ ]:
# Attest the actual Google-hosted GPU runtime. Do not assume GPU class.
runtime_started = datetime.now(timezone.utc).isoformat()

try:
    import google.colab  # type: ignore  # noqa: F401
    colab_present = True
except Exception:
    colab_present = False

if not colab_present:
    blocked("NOT_GOOGLE_COLAB_HOSTED_RUNTIME")

try:
    import torch
except Exception as exc:
    blocked(
        "TORCH_IMPORT_FAILED",
        failure_class=type(exc).__name__,
        failure_message_sha256=sha256_bytes(str(exc).encode("utf-8", errors="replace")),
    )

if not torch.cuda.is_available():
    blocked("CUDA_UNAVAILABLE")

gpu_count = int(torch.cuda.device_count())
expected_count = runtime_policy["allowed_gpu_count"]
if gpu_count != expected_count:
    blocked("GPU_COUNT_POLICY_MISMATCH", observed=gpu_count, expected=expected_count)

gpu_models = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
gpu_total_memory = [int(torch.cuda.get_device_properties(i).total_memory) for i in range(gpu_count)]
allowed_gpu_models = list(runtime_policy["allowed_gpu_models"])
allow_unlisted = runtime_policy["allow_unlisted_gpu_model"]
if not allow_unlisted:
    unexpected = [name for name in gpu_models if name not in allowed_gpu_models]
    if unexpected:
        blocked("GPU_MODEL_NOT_IN_FROZEN_POLICY", observed=gpu_models, allowed=allowed_gpu_models)

print(json.dumps({
    "runtime_provider": "GOOGLE_COLAB",
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "gpu_count": gpu_count,
    "gpu_models": gpu_models,
    "gpu_total_memory_bytes": gpu_total_memory,
}, indent=2))


In [ ]:
# Bind execution to the exact canonical repository commit/tree.
if REPO.exists():
    blocked("REPOSITORY_PATH_ALREADY_EXISTS", path=str(REPO))

network_policy = config["network_policy"]
if network_policy.get("allow_repository_clone") is not True:
    blocked("REPOSITORY_CLONE_NOT_ALLOWED_BY_FROZEN_POLICY")
allowed_hosts = network_policy.get("allowed_hosts")
if not isinstance(allowed_hosts, list) or "github.com" not in allowed_hosts:
    blocked("REPOSITORY_HOST_NOT_ALLOWED_BY_FROZEN_POLICY", required_host="github.com")

clone = subprocess.run(
    ["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/TheHalfMoon/MESC.git", str(REPO)],
    text=True,
    capture_output=True,
)
if clone.returncode != 0:
    blocked(
        "REPOSITORY_CLONE_FAILED",
        returncode=clone.returncode,
        stderr_sha256=sha256_bytes(clone.stderr.encode("utf-8", errors="replace")),
    )

checkout = subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", config["repository_sha"]],
    text=True,
    capture_output=True,
)
if checkout.returncode != 0:
    blocked(
        "REPOSITORY_CHECKOUT_FAILED",
        returncode=checkout.returncode,
        stderr_sha256=sha256_bytes(checkout.stderr.encode("utf-8", errors="replace")),
    )

observed_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
observed_tree = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD^{tree}"], text=True).strip()
if observed_head != config["repository_sha"]:
    blocked("REPOSITORY_SHA_MISMATCH", observed=observed_head, expected=config["repository_sha"])
if observed_tree != config["repository_tree"]:
    blocked("REPOSITORY_TREE_MISMATCH", observed=observed_tree, expected=config["repository_tree"])

print(json.dumps({"repository_sha": observed_head, "repository_tree": observed_tree}, indent=2))


In [ ]:
# Record package name/version metadata only; do not persist direct URLs or credentials.
packages = []
for distribution in importlib.metadata.distributions():
    name = distribution.metadata.get("Name")
    version = distribution.version
    if name and version:
        packages.append({"name": str(name), "version": str(version)})
packages = sorted(packages, key=lambda item: (item["name"].casefold(), item["version"]))

environment_manifest = {
    "schema_version": ENVIRONMENT_SCHEMA,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "packages": packages,
}
environment_sha256 = write_json(EVIDENCE / "environment-manifest.json", environment_manifest)

try:
    transformers_version = importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:
    transformers_version = None

runtime_receipt = {
    "schema_version": RUNTIME_SCHEMA,
    "experiment_config_sha256": config_sha256,
    "repository_sha": observed_head,
    "repository_tree": observed_tree,
    "execution_started_at_utc": runtime_started,
    "execution_completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "runtime_provider": "GOOGLE_COLAB",
    "runtime_class": "GOOGLE_COLAB_HOSTED_GPU_RUNTIME",
    "python_version": platform.python_version(),
    "platform_string": platform.platform(),
    "torch_version": torch.__version__,
    "transformers_version": transformers_version,
    "cuda_available": True,
    "cuda_version": torch.version.cuda,
    "gpu_count": gpu_count,
    "gpu_models": gpu_models,
    "gpu_total_memory_bytes": gpu_total_memory,
    "colab_release_tag_or_image_identity_if_observable": os.environ.get("COLAB_RELEASE_TAG"),
    "installed_environment_manifest_sha256": environment_sha256,
    "network_policy_observation": "FROZEN_PREPARATION_NETWORK_POLICY_OBSERVED",
    "credential_surface_observation": "NO_CREDENTIAL_READ_OR_PERSISTENCE_IN_PREPARATION_TEMPLATE",
    "final_runtime_disposition": "PASS_RUNTIME_PREFLIGHT",
    "stop_reason": None,
}
runtime_sha256 = write_json(EVIDENCE / "runtime-receipt.json", runtime_receipt)
print(json.dumps({
    "runtime_receipt_sha256": runtime_sha256,
    "environment_manifest_sha256": environment_sha256,
}, indent=2))


In [ ]:
# Intentional preparation stop.
# Candidate model acquisition and medical evaluation adapters are not guessed here.
blocked(
    "CANDIDATE_EXECUTION_ADAPTERS_NOT_CANONICAL_YET",
    runtime_receipt_sha256=runtime_sha256,
    note="Preparation template completed frozen-config/runtime/repository attestation only.",
)
